# Experiment No 7

**Aim:** To build Bagging, Boosting, and Stacking ensemble classification models on the Spambase dataset and evaluate their performance using appropriate metrics.

**Dataset:** [Spambase – UCI ML Repository](https://archive.ics.uci.edu/dataset/94/spambase)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier,
    AdaBoostClassifier, GradientBoostingClassifier,
    StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay
)

spambase = fetch_ucirepo(id=94)
df = pd.concat([spambase.data.features, spambase.data.targets], axis=1)
display(df.head(3))

## Preprocessing

In [ ]:
X_data = df.drop(columns=['Class'])
y_data = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42, stratify=y_data
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

## Bagging – BaggingClassifier & Random Forest

In [ ]:
bag = RandomForestClassifier(n_estimators=100, random_state=42)
bag.fit(X_train, y_train)
y_pred_bag = bag.predict(X_test)
y_prob_bag = bag.predict_proba(X_test)[:, 1]

metrics_bag = pd.DataFrame({
    'Accuracy':  [accuracy_score(y_test, y_pred_bag)],
    'Precision': [precision_score(y_test, y_pred_bag)],
    'Recall':    [recall_score(y_test, y_pred_bag)],
    'F1-Score':  [f1_score(y_test, y_pred_bag)],
    'ROC-AUC':   [roc_auc_score(y_test, y_prob_bag)]
}).round(4)
display(metrics_bag)
print(classification_report(y_test, y_pred_bag, target_names=['Not Spam', 'Spam']))

### Confusion Matrix, ROC Curve & Feature Importances – Random Forest

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

sns.heatmap(confusion_matrix(y_test, y_pred_bag), annot=True, fmt='d',
            cmap='Blues', ax=axes[0],
            xticklabels=['Not Spam','Spam'], yticklabels=['Not Spam','Spam'])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix – Random Forest')

RocCurveDisplay.from_predictions(y_test, y_prob_bag, ax=axes[1], name='Random Forest')
axes[1].plot([0,1],[0,1],'k--')
axes[1].set_title('ROC Curve – Random Forest')

importances = pd.Series(bag.feature_importances_, index=X_data.columns).nlargest(10).sort_values()
importances.plot(kind='barh', color='steelblue', ax=axes[2])
axes[2].set_xlabel('Importance')
axes[2].set_title('Top 10 Feature Importances')

plt.tight_layout(); plt.show()

## Boosting – AdaBoost & Gradient Boosting

In [ ]:
ada = AdaBoostClassifier(n_estimators=100, random_state=42)
ada.fit(X_train, y_train)
y_pred_ada = ada.predict(X_test)
y_prob_ada = ada.predict_proba(X_test)[:, 1]

gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
y_prob_gb = gb.predict_proba(X_test)[:, 1]

metrics_boost = pd.DataFrame({
    'Model':     ['AdaBoost', 'Gradient Boosting'],
    'Accuracy':  [accuracy_score(y_test, y_pred_ada),  accuracy_score(y_test, y_pred_gb)],
    'Precision': [precision_score(y_test, y_pred_ada), precision_score(y_test, y_pred_gb)],
    'Recall':    [recall_score(y_test, y_pred_ada),    recall_score(y_test, y_pred_gb)],
    'F1-Score':  [f1_score(y_test, y_pred_ada),        f1_score(y_test, y_pred_gb)],
    'ROC-AUC':   [roc_auc_score(y_test, y_prob_ada),   roc_auc_score(y_test, y_prob_gb)]
}).round(4)
display(metrics_boost)

### Confusion Matrices & ROC Curves – Boosting

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))

for ax, cm, title, cmap in zip(
    [axes[0,0], axes[0,1]],
    [confusion_matrix(y_test, y_pred_ada), confusion_matrix(y_test, y_pred_gb)],
    ['Confusion Matrix – AdaBoost', 'Confusion Matrix – Gradient Boosting'],
    ['Oranges', 'Greens']
):
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=['Not Spam','Spam'], yticklabels=['Not Spam','Spam'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title(title)

RocCurveDisplay.from_predictions(y_test, y_prob_ada, ax=axes[1,0], name='AdaBoost')
axes[1,0].plot([0,1],[0,1],'k--'); axes[1,0].set_title('ROC Curve – AdaBoost')

RocCurveDisplay.from_predictions(y_test, y_prob_gb, ax=axes[1,1], name='Gradient Boosting')
axes[1,1].plot([0,1],[0,1],'k--'); axes[1,1].set_title('ROC Curve – Gradient Boosting')

plt.tight_layout(); plt.show()

## Stacking

In [ ]:
base_learners = [
    ('dt',  DecisionTreeClassifier(max_depth=5, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=5)),
    ('nb',  GaussianNB())
]
meta_learner = LogisticRegression(max_iter=1000, random_state=42)

stack = StackingClassifier(estimators=base_learners, final_estimator=meta_learner, cv=5)
stack.fit(X_train_scaled, y_train)
y_pred_stack = stack.predict(X_test_scaled)
y_prob_stack = stack.predict_proba(X_test_scaled)[:, 1]

metrics_stack = pd.DataFrame({
    'Accuracy':  [accuracy_score(y_test, y_pred_stack)],
    'Precision': [precision_score(y_test, y_pred_stack)],
    'Recall':    [recall_score(y_test, y_pred_stack)],
    'F1-Score':  [f1_score(y_test, y_pred_stack)],
    'ROC-AUC':   [roc_auc_score(y_test, y_prob_stack)]
}).round(4)
display(metrics_stack)
print(classification_report(y_test, y_pred_stack, target_names=['Not Spam', 'Spam']))

### Confusion Matrix & ROC Curve – Stacking

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

sns.heatmap(confusion_matrix(y_test, y_pred_stack), annot=True, fmt='d',
            cmap='Purples', ax=axes[0],
            xticklabels=['Not Spam','Spam'], yticklabels=['Not Spam','Spam'])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix – Stacking')

RocCurveDisplay.from_predictions(y_test, y_prob_stack, ax=axes[1], name='Stacking')
axes[1].plot([0,1],[0,1],'k--')
axes[1].set_title('ROC Curve – Stacking')

plt.tight_layout(); plt.show()

## Conclusion

Ensemble methods consistently outperform single classifiers on Spambase. Bagging (Random Forest) reduces variance, Boosting (AdaBoost, Gradient Boosting) reduces bias by sequentially correcting errors, and Stacking leverages diverse base learners with a meta-learner to further improve predictions.